# Train a linear policy using ARS (with a single agent)

### - still needs to be tested with attacks

In [1]:
#base_dir = "C:/Users/etucker2/Documents/MAGIC/MAGIC_emulator/MAGIC/"
base_dir = "C:/Users/bhtan/Desktop/Projects/MAGIC/"
# base_dir = "C:/Users/danie/Desktop/MAGIC/MAGIC/"

import os
os.chdir(base_dir)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
from random import sample

from magic.simulations.Simulation import OpenDSSSimulation, OpenDSSSimParams
from magic.envs import OpenDSSOscillationEnv
from magic.agents.ars import ARSParams, ARSAgent, AgentParams


# from attack_scripts.attack_wrapper import CyberAttack, CyberAttackWrapper
# from attack_scripts.control_param_attack import get_vvvw_control_params, vvvw_control_param_attack
# from attack_scripts.power_attack import pv_power_setpoint_attack, pv_power_output_attack, pv_power_output_replication_attack, pv_power_setpoint_replication_attack

In [2]:
#Load the load and solar data from csv files in the "data" folder
data_dir = base_dir + "data/"
feeder_filename = data_dir + 'ieee37.dss'
load_filename = data_dir + 'load_data.csv'
solar_filename = data_dir + 'solar_data.csv'
solar_inverter_VVVW_breakpoints = data_dir + 'solar_vv_breakpoints.csv'

s_max_scaling = 1.1
p_max_scaling = 1.0
q_max_scaling = 1.0

#define the power factor to create reactive power loads
power_factor = 0.9

load_scaling_factor = 0.1
solar_scaling_factor = 0.1

vvvw_active = True

params = OpenDSSSimParams(load_filename, solar_filename, feeder_filename,
                          solar_inverter_VVVW_breakpoints, s_max_scaling,
                          p_max_scaling, q_max_scaling, load_scaling_factor,
                          solar_scaling_factor, power_factor, vvvw_active)


In [3]:
env = OpenDSSOscillationEnv(params)

simulation warming up....


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 46.83it/s]

simulation warmup complete


## Define an agent

In [4]:
#create a base_agent
algorithm_type = "ars"
policy_type = "linear"
randomized = False #ignore for now
n_rollouts = 5 #number of times multistep_simulation is run
n_timesteps = 60 #run multistep for 60 seconds (60 seconds per rollout + 5 rollouts -> 5 min simulation time in total)
n_simulations = 2 #number of times agent performs ARS
# (2*number of random directions + 1) -> # of rollouts performed for each iteration of ARS
# (2*number of random directions + 1)*number of simulations -> total number of 5 min simulations completed

base_params = AgentParams(algorithm_type, policy_type, randomized, n_rollouts, n_timesteps, n_simulations)

#define ars hyperparameters
rand_directions = 2
best_directions = 1
learning_rate = 0.002
exploration_noise = 0.02

#create ARS_Agent using the base agent parameters & ars parameters
ars_params1 = ARSParams(*base_params.__dict__.values(), rand_directions, best_directions, learning_rate, exploration_noise)
ars1 = ARSAgent(env, ars_params1) #passing inititalized env to the agent so different agents can be trained with a single env
ars1.get_params()

ARSParams(algorithm='ars', policy='linear', random=False, rollouts=5, episodes=60, simulations=2, rand_directions=2, best_directions=1, learning_rate=0.002, exploration_noise=0.02)

In [5]:
#policy initialized when agent is initialized
ars1.policy.get_policy()

array([[0., 0., 0., 0., 0., 0., 0., 0.]])

## Train agent

In [6]:
policies, rewards = ars1.train()

simulation warming up....


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 41.53it/s]


simulation warmup complete
running simulation for 60 steps starting at t = 0


100%|██████████████████████████████████████████████████████████████████████████████████| 60/60 [00:01<00:00, 46.99it/s]


simulation complete
running simulation for 60 steps starting at t = 60


...

simulation complete
running simulation for 60 steps starting at t = 180


100%|██████████████████████████████████████████████████████████████████████████████████| 60/60 [00:01<00:00, 45.57it/s]


simulation complete
running simulation for 60 steps starting at t = 240


100%|██████████████████████████████████████████████████████████████████████████████████| 60/60 [00:02<00:00, 28.00it/s]


simulation complete


In [7]:
rewards

[np.float64(-2073.6497995070063), np.float64(-2136.552263548004)]

In [8]:
policies

[array([[ 5.48910816e-04, -1.83355277e-03, -1.11530213e-02,
          5.04477569e-03,  5.19340667e-05, -7.19372306e-03,
          4.38721065e-03, -1.24779209e-03]]),
 array([[-0.00125329,  0.00639938, -0.01219829,  0.00448312,  0.00132221,
         -0.00586406,  0.00517387,  0.00538448]])]

In [9]:
M, b = ars1.policy.resolve_policy()
print("M:", M)
print("b:", b)

M: [[-0.00125329  0.00639938 -0.01219829  0.00448312  0.00132221]]
b: [[-0.00586406  0.00517387  0.00538448]]


In [10]:
#to update a policy:
new_policy = policies[0] #change back to first policy that was trained
ars1.policy.update_policy(new_policy)
ars1.policy.get_policy()

array([[ 5.48910816e-04, -1.83355277e-03, -1.11530213e-02,
         5.04477569e-03,  5.19340667e-05, -7.19372306e-03,
         4.38721065e-03, -1.24779209e-03]])

In [11]:
M_new, b_new = ars1.policy.resolve_policy()
print("new M:", M_new)
print("new b:", b_new)

new M: [[ 5.48910816e-04 -1.83355277e-03 -1.11530213e-02  5.04477569e-03
   5.19340667e-05]]
new b: [[-0.00719372  0.00438721 -0.00124779]]


In [12]:
env.V.head(6) #voltages for entire five min simulation

0         1         2         3         4         5    \
bus       phase                                                               
sourcebus 1      1.000007  1.000007  1.000007  1.000007  1.000007  1.000007   
          2      1.000007  1.000007  1.000007  1.000007  1.000007  1.000007   
          3      1.000013  1.000013  1.000013  1.000013  1.000013  1.000013   
799       1      1.038143  1.036096  1.038075  1.036002  1.037969  1.035852   
          2      1.173216  1.171349  1.173072  1.171166  1.172949  1.171062   
          3      0.983751  0.984506  0.984356  0.985024  0.984770  0.985350   

                      6         7         8         9    ...       290  \
bus       phase                                          ...             
sourcebus 1      1.000007  1.000007  1.000007  1.000008  ...  1.000007   
          2      1.000007  1.000007  1.000007  1.000007  ...  1.000007   
          3      1.000013  1.000013  1.000013  1.000013  ...  1.000013   
799       1      1.037791  1.035620  1.037530  1.039595  ...  1.037348   
          2      1.172865  1.170949  1.172704  1.174670  ...  1.177583   
          3      0.985015  0.985551  0.985201  0.984522  ...  0.983154   

                      291       292       293       294       295       296  \
bus       phase                                                               
sourcebus 1      1.000008  1.000007  1.000008  1.000008  1.000007  1.000008   
          2      1.000007  1.000007  1.000007  1.000007  1.000007  1.000007   
          3      1.000013  1.000013  1.000013  1.000013  1.000013  1.000013   
799       1      1.039348  1.037526  1.039372  1.039086  1.037026  1.038942   
          2      1.179495  1.177913  1.179809  1.178474  1.177006  1.178972   
          3      0.982371  0.982651  0.981676  0.984660  0.984551  0.983573   

                      297       298       299  
bus       phase                                
sourcebus 1      1.000007  1.000007  1.000007  
          2      1.000007  1.000007  1.000007  
          3      1.000013  1.000013  1.000013  
799       1      1.036970  1.038744  1.036704  
          2      1.177326  1.179282  1.177614  
          3      0.983731  0.982607  0.982825  

[6 rows x 300 columns]

In [13]:
env.pv_injections.head(6)

0                      1              \
                                       p           q          p           q   
name     load name bus  phase                                                 
S701a_pv S701a     S701 a      10.494811  -12.662580  10.495778  -11.172328   
S701b_pv S701b     S701 b       0.149695  176.898947   0.112271  177.123928   
S701c_pv S701c     S701 c      29.871061  452.166357  29.857490  454.255988   
S712c_pv S712c     S712 c       7.335426  110.895499   7.331968  111.646294   
S713c_pv S713c     S713 c       7.197167  110.802159   7.200345  111.519442   
S714a_pv S714a     S714 a       1.335044    7.306965   1.334161    7.529238   

                                       2                      3              \
                                       p           q          p           q   
name     load name bus  phase                                                 
S701a_pv S701a     S701 a      10.503730  -13.277408  10.510823  -11.774362   
S701b_pv S701b     S701 b       0.084203  177.292664   0.063153  177.419216   
S701c_pv S701c     S701 c      29.846687  455.823253  29.835111  456.998925   
S712c_pv S712c     S712 c       7.331150  112.209276   7.331684  112.631439   
S713c_pv S713c     S713 c       7.201972  112.057452   7.201574  112.461063   
S714a_pv S714a     S714 a       1.333575    7.252499   1.333520    7.466510   

                                       4              ...           295  \
                                       p           q  ...             p   
name     load name bus  phase                         ...                 
S701a_pv S701a     S701 a      10.515447  -13.904753  ...  1.069524e+01   
S701b_pv S701b     S701 b       0.047364  177.514129  ...  2.081037e-38   
S701c_pv S701c     S701 c      29.829827  457.880460  ...  3.100928e+01   
S712c_pv S712c     S712 c       7.331506  112.948098  ...  7.676934e+00   
S713c_pv S713c     S713 c       7.200480  112.763821  ...  6.747693e+00   
S714a_pv S714a     S714 a       1.333603    7.179151  ...  1.389167e+00   

                                                    296              \
                                        q             p           q   
name     load name bus  phase                                         
S701a_pv S701a     S701 a       -5.000667  1.070382e+01   -7.134598   
S701b_pv S701b     S701 b      177.798871  1.560778e-38  177.798871   
S701c_pv S701c     S701 c      460.446119  3.102053e+01  460.445362   
S712c_pv S712c     S712 c      113.875281  7.677583e+00  113.875237   
S713c_pv S713c     S713 c      113.699659  6.756253e+00  113.699151   
S714a_pv S714a     S714 a        8.448948  1.388950e+00    8.150838   

                                        297                       298  \
                                          p           q             p   
name     load name bus  phase                                           
S701a_pv S701a     S701 a      1.071147e+01   -5.706743  1.071786e+01   
S701b_pv S701b     S701 b      1.170583e-38  177.798871  8.779375e-39   
S701c_pv S701c     S701 c      3.102341e+01  460.445169  3.103174e+01   
S712c_pv S712c     S712 c      7.678360e+00  113.875185  7.679128e+00   
S713c_pv S713c     S713 c      6.762777e+00  113.698763  6.770506e+00   
S714a_pv S714a     S714 a      1.389202e+00    8.344797  1.389491e+00   

                                                    299              
                                        q             p           q  
name     load name bus  phase                                        
S701a_pv S701a     S701 a       -7.709495  1.072402e+01   -6.394828  
S701b_pv S701b     S701 b      177.798871  6.584531e-39  177.798871  
S701c_pv S701c     S701 c      460.444608  3.104875e+01  460.443461  
S712c_pv S712c     S712 c      113.875133  7.680755e+00  113.875023  
S713c_pv S713c     S713 c      113.698304  6.779855e+00  113.697746  
S714a_pv S714a     S714 a        8.069616  1.389830e+00    8.252212  

[6 rows x 